# Урок 18 · LLM на практике: анализ отзывов и свой бот

Сегодня решаем НАСТОЯЩУЮ задачу: автоматически определяем тональность отзывов (позитив/негатив)
и делаем бота с характером. Всё — на бесплатных моделях Hugging Face, **без API-ключей**.

> План: 1) готовая модель тональности → 2) свой датасет отзывов → 3) бот с характером → 4) ⭐ мини-RAG.

## ⚠️ Важно: два разных `pipeline` / `Pipeline`

На прошлых уроках у нас был **`Pipeline` из sklearn** — конвейер предобработки и модели (с большой буквы, из `sklearn`).

Сегодня будет **`pipeline` из Hugging Face** — это ДРУГОЕ (с маленькой буквы, из `transformers`).
Это готовый «загрузчик» обученной модели одной строкой: сказал задачу — получил рабочую модель.

Не путай их! Это разные инструменты с похожим названием.

| Что | Откуда | Зачем |
|---|---|---|
| `Pipeline` (sklearn) | `from sklearn.pipeline import Pipeline` | склеить свою предобработку + модель |
| `pipeline` (HF) | `from transformers import pipeline` | загрузить чужую готовую модель |

## Шаг 1. Загружаем готовую модель тональности

Одна строка — и у нас модель, обученная различать позитив и негатив. Ничего обучать не надо.

In [ ]:
!pip install transformers -q

In [ ]:
from transformers import pipeline

# создаём готовую модель для анализа тональности (задача "sentiment-analysis")
# HF сам скачает подходящую модель из интернета
sentiment = pipeline("sentiment-analysis",
                     model="distilbert-base-uncased-finetuned-sst-2-english")

# пробуем на одном отзыве
result = sentiment("This movie was absolutely wonderful, I loved it!")
print(result)

**❓ Вопрос 1.** Модель вернула `label` и `score`. Что означает каждое?
`score` — это уверенность модели от 0 до 1. Что значит score = 0.99?

## Шаг 2. Проверяем на нескольких отзывах

Прогоним пачку отзывов и посмотрим, где модель уверена, а где сомневается.

In [ ]:
reviews = [
    "Best purchase ever, works perfectly!",
    "Terrible quality, broke after one day.",
    "It's okay, nothing special.",
    "I am so happy with this product!",
    "Waste of money, do not buy.",
]

for r in reviews:
    out = sentiment(r)[0]
    print(f"{out['label']:8} ({out['score']:.2f})  <-  {r}")

**❓ Вопрос 2.** Найди отзыв, где модель менее уверена (score ближе к 0.5–0.7).
Почему именно он? Что в нём «сбивает» модель?

## Шаг 3. Свой датасет отзывов + подсчёт

Собери 5–6 СВОИХ отзывов (о чём угодно: игра, кафе, фильм). Модель разложит их по тональности.

In [ ]:
my_reviews = [
    "Эта игра затягивает на часы, обожаю её!",     # можешь заменить на свои
    "Ужасный сервис, больше не приду.",
    "Неплохо, но дороговато.",
    "Лучшее кафе в городе!",
]

# ВАЖНО: модель выше обучена на английском. Для русского возьмём многоязычную модель:
multi = pipeline("sentiment-analysis",
                 model="nlptown/bert-base-multilingual-uncased-sentiment")

pos, neg = 0, 0
for r in my_reviews:
    out = multi(r)[0]
    stars = int(out["label"][0])   # эта модель отвечает "1 star".."5 stars"
    verdict = "ПОЗИТИВ" if stars >= 4 else ("НЕГАТИВ" if stars <= 2 else "НЕЙТРАЛ")
    print(f"{verdict:8} {stars}★  <-  {r}")
    if stars >= 4: pos += 1
    elif stars <= 2: neg += 1

print(f"\nИтого: позитивных {pos}, негативных {neg}")

## Шаг 4. Бот с характером

Теперь генеративная модель — она не классифицирует, а ПРОДОЛЖАЕТ текст.
Мы зададим боту роль (характер) — это и есть промптинг.

In [ ]:
generator = pipeline("text-generation", model="distilgpt2")

# промпт задаёт РОЛЬ и стиль. Модель продолжит в этом духе.
prompt = "You are a cheerful sports coach. Advice for a beginner runner:"

out = generator(prompt, max_new_tokens=40, num_return_sequences=1)
print(out[0]["generated_text"])

**❓ Вопрос 3.** Поменяй роль в промпте (например «строгий учитель математики» или «пират»).
Как меняется стиль ответа? Это и есть промпт-инжиниринг: роль настраивает модель.

> Заметь: distilgpt2 — маленькая модель, она часто говорит бессвязно и может галлюцинировать (помнишь урок 17?). Большие модели вроде Claude отвечают куда лучше — но принцип тот же.

## Шаг 5. Три приёма хорошего промпта (из учебника)

1. **Дай роль:** «Ты — опытный редактор...»
2. **Дай примеры:** покажи 1–2 образца ответа.
3. **Проси формат:** «ответь одним словом: позитив или негатив».

Сравним плохой и хороший промпт на классификации.

In [ ]:
# плохой промпт: расплывчатый
bad = generator("Отзыв: еда была холодной. Что думаешь?", max_new_tokens=30)
print("ПЛОХОЙ промпт:\n", bad[0]["generated_text"])

print("\n" + "="*50 + "\n")

# хороший промпт: роль + чёткая задача + формат
good = generator("You classify reviews. Answer only 'positive' or 'negative'.\nReview: the food was cold.\nAnswer:", max_new_tokens=5)
print("ХОРОШИЙ промпт:\n", good[0]["generated_text"])

---
## 🎯 Задания

### 🟢 Базовый
Собери 8 своих отзывов, прогони через модель тональности (шаг 3), посчитай позитив/негатив. Найди отзыв, где модель ошиблась, и объясни почему.

### 🟡 Продвинутый
Сделай бота с ярким характером через роль в промпте. Придумай 3 разные роли и сравни, как меняются ответы на один и тот же вопрос.

### ⭐ Со звёздочкой (мини-RAG)
RAG-lite: дай модели проверенный текст и заставь отвечать ТОЛЬКО по нему — так уменьшают галлюцинации. Заготовка ниже.

In [ ]:
qa = pipeline("question-answering",
              model="distilbert-base-cased-distilled-squad")

# наш проверенный текст (контекст). Модель ответит ТОЛЬКО по нему.
context = '''Палмер-пингвины живут в Антарктике. Существует три вида:
Adelie, Chinstrap и Gentoo. Gentoo — самые крупные из трёх.'''

question = "Какой вид пингвинов самый крупный?"
answer = qa(question=question, context=context)
print("Вопрос:", question)
print("Ответ модели:", answer["answer"], f"(уверенность {answer['score']:.2f})")

# попробуй задать вопрос, ответа на который НЕТ в тексте — что вернёт модель?

## Мини-итог

- Hugging Face `pipeline` (маленькая буква) — это ..., а sklearn `Pipeline` (большая буква) — это ...
- Промпт — это ..., и три приёма хорошего промпта: ..., ..., ...
- RAG уменьшает галлюцинации, потому что ...

> Ты решил реальную задачу бизнеса — автоматический разбор отзывов — на бесплатных моделях. Это не «ещё один чат-бот».